# Lab 7: Customer Segmentation

**Task:** group a bank's customers into segments with similar profiles, without any target to learn from (unsupervised learning).

**Dataset:** Bank Marketing (UCI: https://archive.ics.uci.edu/dataset/222/bank+marketing), included as `data/bank_marketing.zip` as downloaded: 45,211 customers of a Portuguese bank contacted in a phone campaign for term deposits, with their age, job, marital status, education, account balance, loans, and details of the campaign contacts.

**What we will do:**
1. Clean the data and choose the features that describe the *customer*.
2. Standardise the numeric features (distances need comparable scales).
3. Run K-means for $k = 2$ to $10$: SSE (elbow) curve and silhouette score; and K-prototypes, a variant made for mixed numeric and categorical data.
4. Reduce the data to 2 dimensions with PCA to visualise the clusters, and profile each cluster.
5. Recommend a number of clusters and write a "persona card" for each.

In [ ]:
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 20)

outer = zipfile.ZipFile("data/bank_marketing.zip")
print(outer.namelist())
inner = zipfile.ZipFile(io.BytesIO(outer.read("bank.zip")))
print(inner.namelist())
bank = pd.read_csv(inner.open("bank-full.csv"), sep=";")     # this file uses ; as the separator
print(bank.shape)
bank.head()

`bank-names.txt` describes the columns. Two kinds:
- **About the customer:** age, job, marital, education, default (credit in default?), balance (average yearly balance, euros), housing (housing loan?), loan (personal loan?).
- **About the campaign:** contact type, day and month of the last call, duration of the call, number of calls in this campaign (campaign), days since a previous campaign contact (pdays, −1 = never), number of previous contacts (previous), outcome of the previous campaign (poutcome), and **y**: did the customer subscribe to a term deposit?

## 1. Cleaning and choosing features

In [ ]:
print("missing values (NaN):", bank.isna().sum().sum())
print("'unknown' values per categorical column:")
print((bank.select_dtypes("object") == "unknown").mean().round(3)[lambda s: s > 0])

There are no NaNs, but **missing values are coded as the text `"unknown"`**: 0.6% of jobs, 4% of education values, 29% of contact types and 82% of previous-campaign outcomes (most customers were never contacted before). For the customer features, the unknowns are few: we keep them as their own category (dropping 4% of customers for an unknown education would throw away good information about everything else). 

There is no ID column in this file (other versions of the data have one; an ID must never be used in clustering, since it carries no information about the customer but would affect distances).

**Which features?** Segments should describe *customers*, so we use the customer attributes. The campaign columns describe how *this campaign* contacted them (which day, how long the call lasted) and `y` is the campaign's outcome; we keep those out of the clustering and use `y` afterwards to see whether the segments differ in how they responded. That is a useful check that the segments mean something.

In [ ]:
numeric_cols = ["age", "balance"]
categorical_cols = ["job", "marital", "education", "default", "housing", "loan"]
customers = bank[numeric_cols + categorical_cols].copy()
print(customers.describe().round(0))

`balance` is very skewed (median 448, maximum 102,127) and can be negative. In a distance-based method, a few customers with huge balances would dominate. A **signed log** transform, $\text{sign}(x)\log(1 + |x|)$, keeps the sign and compresses the extremes.

In [ ]:
customers["balance_slog"] = np.sign(customers["balance"]) * np.log1p(customers["balance"].abs())
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(customers["balance"], bins=100); axes[0].set_title("balance")
axes[1].hist(customers["balance_slog"], bins=100); axes[1].set_title("signed log of balance")
plt.show()

## 2. Standardising and encoding

K-means measures similarity by Euclidean distance. Age varies by about 11 years, the signed-log balance by a few units: without standardising, the feature with the larger numbers decides everything. We standardise the numeric features (mean 0, standard deviation 1) and one-hot encode the categories (each category becomes a 0/1 column).

One-hot columns enter the distance too: two customers with different jobs differ by 1 in two columns, a squared distance of 2. Whether that's "a lot" compared with age differences is a choice, not a fact; this is one of the difficulties of clustering mixed data, and the reason for K-prototypes below.

In [ ]:
from sklearn.preprocessing import StandardScaler

num_std = pd.DataFrame(StandardScaler().fit_transform(customers[["age", "balance_slog"]]), columns=["age", "balance_slog"])
cat_dummies = pd.get_dummies(customers[categorical_cols], dtype=float)
X = pd.concat([num_std, cat_dummies], axis=1)
print(X.shape)

## 3. K-means for k = 2 to 10

**SSE** (sum of squared errors, also called inertia) is the total squared distance from each customer to their cluster's centre; it always falls as $k$ grows, and we look for an "elbow" where it stops falling quickly. The **silhouette score** compares, for each customer, the average distance to their own cluster ($a$) with the average distance to the nearest other cluster ($b$): $s = (b - a)/\max(a, b)$, between −1 and 1, higher meaning tighter, better-separated clusters. It needs all pairwise distances, so we compute it on a random sample of 5,000 customers.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    sil = silhouette_score(X, km.labels_, sample_size=5000, random_state=0)
    rows.append({"k": k, "SSE": km.inertia_, "silhouette": sil, "smallest cluster": np.bincount(km.labels_).min()})
kmeans_table = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(kmeans_table["k"], kmeans_table["SSE"], "o-"); axes[0].set_title("SSE (elbow curve)"); axes[0].set_xlabel("k")
axes[1].plot(kmeans_table["k"], kmeans_table["silhouette"], "o-"); axes[1].set_title("silhouette"); axes[1].set_xlabel("k")
plt.show()
kmeans_table.round(3)

### K-prototypes: built for mixed data

K-prototypes (from the `kmodes` package: `pip install kmodes`) keeps the categorical columns as categories. Each cluster has a prototype: the **mean** of each numeric feature and the **mode** (most common value) of each categorical one. The distance from a customer to a prototype is the squared Euclidean distance on the numeric features plus $\gamma$ times the number of categorical features that differ from the prototype's modes. By default $\gamma$ is set to half the average standard deviation of the numeric features. Its **cost** (the total of these distances) plays the role of SSE. It is slower than K-means, so we run it on a sample of 8,000 customers.

In [ ]:
from kmodes.kprototypes import KPrototypes

sample = customers.sample(8000, random_state=0)
kp_data = pd.concat([pd.DataFrame(StandardScaler().fit_transform(sample[["age", "balance_slog"]]), columns=["age", "balance_slog"], index=sample.index),
                     sample[categorical_cols]], axis=1).values
costs = []
for k in range(2, 9):
    kp = KPrototypes(n_clusters=k, init="Cao", n_init=3, random_state=0).fit(kp_data, categorical=list(range(2, 8)))
    costs.append(kp.cost_)
plt.plot(range(2, 9), costs, "o-"); plt.title("K-prototypes cost"); plt.xlabel("k"); plt.show()
print({k: round(float(c)) for k, c in zip(range(2, 9), costs)})

### Choosing k

Neither curve has a sharp elbow, which is typical of real customer data: people form a continuum rather than neat groups. The silhouette scores are low for every $k$ (0.13 to 0.18) and peak at $k = 3$, but the differences between $k = 2, 3, 4$ are small. But the number of segments is also a *business* choice: 2 segments say little, 10 are too many for a marketing team to act on. We look at $k = 4$, a usable number where the SSE curve has started to flatten, and check whether the clusters are interpretable and differ in behaviour. (Try $k = 3$ and $k = 5$ and compare.)

In [ ]:
k = 4
km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
customers["cluster"] = km.labels_
print(customers["cluster"].value_counts().sort_index().to_dict())

## 4. PCA view and cluster profiles

PCA finds the directions of greatest variance in the (standardised, one-hot) data; projecting onto the first two gives the best 2-D picture of the data in that sense. Clusters that overlap in 2-D may still be separated in the other dimensions.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X)
Z = pca.transform(X)
print("variance explained by the 2 components:", pca.explained_variance_ratio_.round(3), "total", pca.explained_variance_ratio_.sum().round(3))
plt.figure(figsize=(7, 5))
idx = np.random.default_rng(0).choice(len(X), 6000, replace=False)
sc = plt.scatter(Z[idx, 0], Z[idx, 1], c=km.labels_[idx], s=4, cmap="tab10")
plt.legend(*sc.legend_elements(), title="cluster"); plt.xlabel("PC1"); plt.ylabel("PC2"); plt.show()
print("PC loadings (largest):")
loadings = pd.DataFrame(pca.components_.T, index=X.columns, columns=["PC1", "PC2"])
print(loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(6).round(2))

### Profiles
Numeric features: the cluster means (and medians, since balance is skewed). Categorical features: the share of each cluster in the most telling categories. Finally the **subscription rate** (`y`), which the clustering never saw.

In [ ]:
bank["cluster"] = customers["cluster"]
profile = bank.groupby("cluster").agg(customers=("age", "size"), mean_age=("age", "mean"), median_balance=("balance", "median"),
                                      housing_loan=("housing", lambda s: (s == "yes").mean()), personal_loan=("loan", lambda s: (s == "yes").mean()),
                                      married=("marital", lambda s: (s == "married").mean()), single=("marital", lambda s: (s == "single").mean()),
                                      tertiary_edu=("education", lambda s: (s == "tertiary").mean()),
                                      subscribed=("y", lambda s: (s == "yes").mean()))
profile.round(2)

In [ ]:
top_jobs = pd.crosstab(bank["cluster"], bank["job"], normalize="index").round(2)
top_jobs[["management", "blue-collar", "technician", "admin.", "services", "retired", "student"]]

## 5. Recommended number of clusters and persona cards

**Recommendation: 4 clusters.** The SSE and cost curves flatten gradually from about 4 on, the silhouette scores are low for every $k$ (the data has no sharp natural groups), and 4 segments are distinct in age, job, education, loans and, importantly, in how they responded to the campaign, which the clustering never saw. That last point is the strongest evidence that the segments capture something real.

The persona cards below are written from the profile tables; when you rerun with another $k$ or random seed, the cluster numbers change, so read the tables and adapt the cards.

In [ ]:
for c, row in profile.iterrows():
    jobs_c = top_jobs.loc[c].sort_values(ascending=False).head(2)
    print(f"--- Cluster {c}: {int(row['customers'])} customers ({row['customers'] / len(bank):.0%})")
    print(f"    age about {row['mean_age']:.0f}; median balance {row['median_balance']:.0f} euros")
    print(f"    mostly {jobs_c.index[0]} ({jobs_c.iloc[0]:.0%}) and {jobs_c.index[1]} ({jobs_c.iloc[1]:.0%}); "
          f"tertiary education {row['tertiary_edu']:.0%}; married {row['married']:.0%}")
    print(f"    housing loan {row['housing_loan']:.0%}, personal loan {row['personal_loan']:.0%}; "
          f"subscribed to the term deposit: {row['subscribed']:.1%} (overall {(bank['y'] == 'yes').mean():.1%})")

### Persona cards (from the run above, random_state=0)

**Cluster 0: "Young professionals"** (21%). About 35, almost all university-educated, mostly managers and technicians, half single. Moderate balances, fewer loans than average. **Most responsive to the term-deposit campaign (16% subscribed).** Approach: digital channels, investment and savings products, career-stage offers.

**Cluster 1: "Young working homeowners"** (35%, the largest). About 35, no university degree, many blue-collar, technician, admin and services jobs. Most have a housing loan (71%). Below-average response (10%). Approach: products that fit around a mortgage: payment protection, small regular-saving plans.

**Cluster 2: "Established older customers"** (29%). About 54, mostly married, including most of the retired customers; fewer housing loans (37%), the highest median balance. Above-average response (13%). Approach: term deposits and pension-oriented products; branch or phone contact.

**Cluster 3: "Stretched borrowers"** (15%). About 39, **negative median balance**, the highest share of personal loans (26%) and many housing loans. **Least responsive (6%).** Approach: don't spend campaign calls selling deposits here; offer debt consolidation or budgeting help instead.

Two cautions. The segments are a *useful summary*, not a natural law: with another $k$, another feature choice or a different weighting of the categories, the boundaries move. And the response rates come from one campaign; check the segments against other outcomes before building strategy on them.